# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My lane is Refresh / Content Opportunity Scoring. The decision is to rank content pages so that a reviewer can inspect the highest-priority pages first.

I will start with Logistic Regression because the dataset contains a binary performance outcome proxy based on `trend_direction`. Logistic Regression is a simple and interpretable supervised model that can produce a probability score for each page, which can then be used to rank the pages.

The model will be compared against the frozen Week-4 rule-based baseline. The purpose is not to use ML simply because it is more complex, but to test whether a learned combination of decision-time features can improve the ranking produced by the existing rule.

I will use Precision@K as the main evaluation metric because the real decision is a ranked review queue: reviewers can only inspect a limited number of pages. I will use the same K values and evaluation population when comparing the model with the Week-4 baseline.

The target is a decline proxy derived from `trend_direction`. `trend_direction` itself will not be used as a feature because it is the source of the outcome being predicted. `trend_pct` will also be excluded from the feature set for the same reason.

In [1]:
import pandas as pd
import numpy as np

SEED = 42

DATA_PATH = "C:/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)


print("Dataset Shape: ",df.shape)
print("\nColumns:")
print(df.columns.tolist())
df.head()

Dataset Shape:  (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [2]:
df['is_declining'] = (
  df["trend_direction"] == "down"
).astype(int)
print(df["is_declining"].value_counts())
print("\nTarget Rate:")
print(df['is_declining'].mean())

is_declining
1    16262
0    13738
Name: count, dtype: int64

Target Rate:
0.5420666666666667


In [3]:
pd.crosstab(
  df['trend_direction'],
  df['is_declining']
)

is_declining,0,1
trend_direction,,
down,0,16262
flat,1152,0
new,2236,0
stable,5962,0
up,4388,0


In [4]:
candidate_features = [
  "days_since_last_update",
  "impressions_90d",
  "search_volume"
]

X = df[candidate_features].copy()
y = df["is_declining"].copy()

print("Features: ")
print(X.columns.tolist())
print("\nTarget: ")
print(y.name)
print("\nFeature Shape: ",X.shape)
print("\nTarget Shape: ",y.shape)

Features: 
['days_since_last_update', 'impressions_90d', 'search_volume']

Target: 
is_declining

Feature Shape:  (30000, 3)

Target Shape:  (30000,)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a client-level holdout split rather than a random row split. The dataset contains multiple pages from the same client, so randomly splitting rows could place pages from the same client in both training and test data and make the evaluation artificially easy.

The split uses `client_id` as the grouping variable. Approximately 80% of clients are used for training and 20% are held out for testing. No client is allowed to appear in both sets.

The test set is therefore intended to measure whether the model can generalize its learned relationship to pages from clients it did not see during training.

I will verify the split by checking that the intersection of train and test client IDs is empty. I will also compare the decline rate across the overall dataset, training set, and test set to identify any substantial distribution difference.

The evaluation population and split will remain fixed when comparing the Logistic Regression model with the frozen Week-4 baseline.

In [5]:
from sklearn.model_selection import GroupShuffleSplit

SEED = 42

gss = GroupShuffleSplit(
  n_splits = 1,
  test_size = 0.20,
  random_state = SEED
)

train_idx, test_idx = next(
  gss.split(
    df,
    y = df['is_declining'],
    groups = df["client_id"]
  )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train shape: ", train_df.shape)
print("Test shape: ", test_df.shape)

Train shape:  (23837, 45)
Test shape:  (6163, 45)


In [6]:
train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])

overlap = train_clients.intersection(test_clients)

print("Train Clients: ", len(train_clients))
print("Test Clients: ",len(test_clients))
print("Client Overlap: ", len(overlap))

assert len(overlap) == 0, "Client leakage detected!"

Train Clients:  25
Test Clients:  7
Client Overlap:  0


In [7]:
print("Overall decline rate:")
print(df["is_declining"].mean())

print("\nTrain decline rate:")
print(train_df["is_declining"].mean())

print("\nTest decline rate:")
print(test_df["is_declining"].mean())

Overall decline rate:
0.5420666666666667

Train decline rate:
0.5501111717078492

Test decline rate:
0.5109524582184002


In [8]:
split_summary = pd.DataFrame({
    "rows": [
        len(df),
        len(train_df),
        len(test_df)
    ],
    "clients": [
        df["client_id"].nunique(),
        train_df["client_id"].nunique(),
        test_df["client_id"].nunique()
    ],
    "decline_rate": [
        df["is_declining"].mean(),
        train_df["is_declining"].mean(),
        test_df["is_declining"].mean()
    ]
}, index=["overall", "train", "test"])

split_summary

,rows,clients,decline_rate
overall,30000,32,0.542067
train,23837,25,0.550111
test,6163,7,0.510952


In [9]:
print("Train clients:")
print(sorted(train_clients))

print("\nTest clients:")
print(sorted(test_clients))

Train clients:
['client_02d20bbd7e', 'client_0b918943df', 'client_19581e27de', 'client_1a6562590e', 'client_25fc0e7096', 'client_2c624232cd', 'client_349c41201b', 'client_3fdba35f04', 'client_4ec9599fc2', 'client_4fc82b26ae', 'client_6208ef0f77', 'client_624b60c58c', 'client_7f2253d7e2', 'client_8722616204', 'client_9400f1b21c', 'client_98a3ab7c34', 'client_9f14025af0', 'client_a88a7902cb', 'client_b4944c6ff0', 'client_bbb965ab0c', 'client_d029fa3a95', 'client_d4735e3a26', 'client_d59eced1de', 'client_e29c9c180c', 'client_f74efabef1']

Test clients:
['client_434c9b5ae5', 'client_4e07408562', 'client_8527a891e2', 'client_8b940be7fb', 'client_bdd2d3af3a', 'client_e629fa6598', 'client_f369cb89fc']


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [10]:
feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "pageviews_90d",
    "users_90d",
    "engaged_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

X_train = train_df[feature_cols]
y_train = train_df["is_declining"]

X_test = test_df[feature_cols]
y_test = test_df["is_declining"]

In [11]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (23837, 20)
X_test: (6163, 20)
y_train: (23837,)
y_test: (6163,)


In [12]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
  ("imputer", SimpleImputer(
    strategy = "median",
    add_indicator = True
  )),

  ("scaler", StandardScaler()),

  ("classifier", LogisticRegression(
    max_iter = 1000,
    random_state = 42
  ))
])

In [13]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](20,)","['impressions_90d','clicks_90d','sessions_90d',...,'engagement_rate', 'scroll_rate','ai_traffic_pct']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,20
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",True
,"missing_values missing_values: int, float, str, np.nan, None or pandas.

In [20]:
test_probs = model.predict_proba(X_test)[: ,1]

test_results = test_df[[
  "content_id",
  "client_id",
  "is_declining"
]].copy()

test_results["decline_probability"] = test_probs

test_results = test_results.sort_values(
  "decline_probability",
  ascending = False
)
test_results.head(10)

,content_id,client_id,is_declining,decline_probability
18063,content_b08562686d22,client_f369cb89fc,1,0.910564
3626,content_8ede62882d0b,client_f369cb89fc,1,0.908012
1537,content_a928cb66d230,client_f369cb89fc,1,0.901100
10175,content_374e795aab68,client_f369cb89fc,0,0.895327
27993,content_26d48a980581,client_f369cb89fc,0,0.886878
14741,content_2bc3b7c8b3d9,client_f369cb89fc,1,0.884278
8016,content_c94a53e3bfb8,client_f369cb89fc,0,0.881281
3195,content_b5e9e6453511,client_f369cb89fc,1,0.880531
23346,content_96dba8ca02c1,client_f369cb89fc,1,0.879457
2646,content_87c007fb5c26,client_f369cb89fc,1,0.876323


In [15]:
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

p20 = precision_at_k(
    test_probs,
    y_test,
    20
)

p50 = precision_at_k(
    test_probs,
    y_test,
    50
)

print("Precision@20:", p20)
print("Precision@50:", p50)

Precision@20: 0.85
Precision@50: 0.84


In [16]:
base_rate = y_test.mean()

print("Test base rate:", base_rate)
print("Precision@20:", p20)
print("Precision@50:", p50)

Test base rate: 0.5109524582184002
Precision@20: 0.85
Precision@50: 0.84


reproducing the frozen Week-4 baseline on the Week-5 test set

In [24]:
baseline_test = test_df.copy()

baseline_test["stale"] = (
  baseline_test["days_since_last_update"] >= 180
).astype(int)

baseline_test["visible"] = (
  baseline_test["impressions_90d"] >= 500
).astype(int)

baseline_test["high_value"] = (
  baseline_test["search_volume"] >= 500
)

baseline_test["baseline_score"] = (
  baseline_test["stale"]*3 + 
  baseline_test["visible"]*2 +
  baseline_test["high_value"]
)

baseline_test[[
  "content_id",
  "client_id",
  "baseline_score",
  "is_declining"
]].head()

,content_id,client_id,baseline_score,is_declining
0,content_304f48230142,client_f369cb89fc,2,1
1,content_a1fb4e703a9e,client_4e07408562,2,1
5,content_d4084a4bc775,client_f369cb89fc,3,1
13,content_a5a2fbc76336,client_8527a891e2,0,0
19,content_af865035b328,client_f369cb89fc,0,1


In [29]:
baseline_p20 = precision_at_k(
    baseline_test["baseline_score"].values,
    baseline_test["is_declining"].values,
    20
)

baseline_p50 = precision_at_k(
    baseline_test["baseline_score"].values,
    baseline_test["is_declining"].values,
    50
)

print("Baseline Precision@20:", baseline_p20)
print("Baseline Precision@50:", baseline_p50)

Baseline Precision@20: 0.4
Baseline Precision@50: 0.4


### baseline vs model
#### - same split,same metrics, plus base rate

In [31]:
comparison = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "Precision@20": [
        baseline_p20,
        p20
    ],
    "Precision@50": [
        baseline_p50,
        p50
    ],
    "Test base rate": [
        y_test.mean(),
        y_test.mean()
    ]
})

comparison

,Method,Precision@20,Precision@50,Test base rate
0,Week-4 baseline,0.40,0.40,0.510952
1,Logistic Regression,0.85,0.84,0.510952


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

- The Logistic Regression model is primarily being evaluated as a ranking model because the decision is to identify which pages should be reviewed first. Therefore, Precision@K remains the main performance measure rather than the 0.50 classification threshold.

- The held-out test set contains 6,163 rows. At a 0.50 probability threshold, the model produced 3,427 correct predictions, 1,798 false positives, and 938 false negatives. These classification errors are useful for understanding model limitations, but they do not replace the Precision@20 and Precision@50 ranking results.

### 4.1 Where is the model wrong?

- The model produces substantially more false positives than false negatives at the 0.50 classification threshold.

- There are 1,798 false positives and 938 false negatives in the held-out test set. The false positives show that the model can assign high decline probabilities to pages whose observed label is not declining.

- Several high-confidence false positives have relatively recent update dates. For example, one page received a decline probability of 0.895 despite having only 20 days since its last update. This suggests that some pages can resemble declining pages according to the available feature patterns even when the observed label is negative.

- The false negatives show a different difficulty. Several pages with a true decline label received very low predicted probabilities. These examples have very low historical impressions, such as 6, 13, or 23 impressions. This indicates that low-volume pages may be difficult for the model to distinguish reliably from noise or other low-signal cases.

In [43]:
test_results = test_df.copy()

test_results["decline_probability"] = test_probs

test_results["predicted_decline"] = (
  test_results["decline_probability"] >= 0.50
).astype(int)

test_results["error_type"] = "correct"

test_results.loc[
  (test_results["predicted_decline"] == 1) &
  (test_results["is_declining"] == 0),
  "error_type"
] = "false_positive"

test_results.loc[
  (test_results["predicted_decline"] == 0) &
  (test_results["is_declining"] == 1),
  "error_type"
] = "false_negative"

test_results["error_type"].value_counts()

error_type
correct           3427
false_positive    1798
false_negative     938
Name: count, dtype: int64

#### False Positives

In [34]:
false_positives = (
  test_results[
    test_results["error_type"] == "false_positive"
  ].sort_values("decline_probability", ascending = False)
)

false_positives[
  [
    "decline_probability",
    "is_declining",
    "impressions_90d",
    "clicks_90d",
    "search_volume",
    "days_since_last_update"
  ]
].head(10)

,decline_probability,is_declining,impressions_90d,clicks_90d,search_volume,days_since_last_update
10175,0.895327,0,235,2,880.0,20
27993,0.886878,0,1266,0,0.0,106
8016,0.881281,0,2164,5,0.0,20
22919,0.861434,0,1038,2,10.0,8
11202,0.856782,0,546,4,0.0,20
26614,0.855210,0,290,0,10.0,20
16052,0.847248,0,873,2,40.0,8
20295,0.845062,0,645,1,10.0,20
17602,0.843796,0,4650,4,10.0,20
19321,0.842225,0,2346,3,0.0,20


#### False Negatives

In [35]:
false_negatives = (
  test_results[
    test_results["error_type"] == "false_negative"
  ].sort_values("decline_probability",ascending = True)
)

false_negatives[
  [
    "decline_probability",
    "is_declining",
    "impressions_90d",
    "clicks_90d",
    "search_volume",
    "days_since_last_update"
  ]
].head(10)

,decline_probability,is_declining,impressions_90d,clicks_90d,search_volume,days_since_last_update
20604,0.088775,1,6,0,NaN,20
4147,0.089398,1,13,0,NaN,20
13922,0.090112,1,13,0,NaN,20
12837,0.094342,1,23,0,NaN,22
20882,0.096498,1,11,0,NaN,22
24089,0.096928,1,57,0,NaN,22
24849,0.098306,1,17,0,0.0,20
727,0.107716,1,14,0,NaN,20
10291,0.108798,1,27,0,NaN,20
1625,0.126699,1,93,0,NaN,22


### 4.2 What does the model lean on?

- Permutation importance was calculated on the held-out test set using Precision@50 as the scoring metric.

- The strongest average permutation importance was observed for `days_with_impressions` (0.302), followed by `scroll_rate` (0.216), `avg_position` (0.184), `content_age_days` (0.182), and `sessions_90d` (0.160).

- This means that shuffling these features caused larger decreases in the model's Precision@50 during the permutation test, indicating that the trained model relies on these signals for its ranking.

- This should be interpreted as model reliance rather than causation. A high permutation importance does not mean that the feature causes content to decline.

- The standard deviations also show that the importance estimates are not equally stable. In particular, `scroll_rate` has relatively high variation compared with its mean importance, so its exact position in the importance ranking should not be treated as definitive.

In [36]:
from sklearn.inspection import permutation_importance

def precision_at_50_scorer(estimator, X, y):
  probabilities = estimator.predict_proba(X)[:, 1]
  return precision_at_k(probabilities, y, 50)

perm_result = permutation_importance(
  model,
  X_test,
  y_test,
  scoring = precision_at_50_scorer,
  n_repeats = 10,
  random_state = SEED,
  n_jobs = -1
)

In [44]:
importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
})

importance_df = (
    importance_df
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

importance_df.head(10)

,feature,importance_mean,importance_std
0,days_with_impressions,0.302,0.062897
1,scroll_rate,0.216,0.085697
2,avg_position,0.184,0.026533
3,content_age_days,0.182,0.039446
4,sessions_90d,0.160,0.053666
5,days_with_sessions,0.148,0.036000
6,days_since_last_update,0.094,0.035833
7,word_count,0.090,0.050000
8,char_count,0.066,0.032311
9,users_90d,0.050,0.013416


### 4.3 Feature interpretation

The most important features are broadly plausible for the content-refresh lane.

- `days_with_impressions` represents how consistently a page has received search visibility and can distinguish pages with sustained exposure from pages with very little history.
- `scroll_rate` provides an engagement signal and may help distinguish pages with different levels of user interaction.
- `avg_position` represents search visibility and ranking position, which is directly relevant to identifying pages whose search performance may require review.
- `content_age_days` represents how old the content is and is relevant to the refresh decision.
- `sessions_90d` provides a measure of recent traffic and exposure.

These interpretations describe relationships used by the model and should not be interpreted as causal claims about why a page declines.

### 4.4 Three Concrete wrong cases

#### Wrong case 1 — High-confidence false positive

The model assigned a decline probability of approximately 0.895, but the observed label was 0. The page had 235 impressions, 2 clicks, a search volume of 880, and only 20 days since its last update.

This is a difficult case because the model considered the available signals consistent with decline even though the observed outcome was negative. The relatively recent update date also shows why a high model score should not automatically be treated as a refresh recommendation.

In [38]:
case_fp = false_positives.head(1)
case_fp

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining,decline_probability,predicted_decline,error_type
10175,content_374e795aab68,client_f369cb89fc,880.0,1.0,HIGH,1.67,keyword article,commercial,2675.0,15908.0,...,300.0,0.0,low,page_3_5,stable,0.0,0,0.895327,1,false_positive


#### Wrong case 2 — Low-volume false negative

The model assigned a decline probability of approximately 0.089, but the observed label was 1. The page had only 6 impressions and 0 clicks.

This is a difficult case because the page has very little historical search activity from which the model can distinguish a genuine decline from a low-volume or noisy page.

In [39]:
case_fn = (
    false_negatives
    .sort_values("decline_probability", ascending=True)
    .head(1)
)

case_fn

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining,decline_probability,predicted_decline,error_type
20604,content_3d85651f289d,client_e629fa6598,NaN,NaN,NaN,NaN,keyword article,commercial,NaN,NaN,...,0.0,0.0,low,striking,down,-100.0,1,0.088775,0,false_negative


#### Wrong case 3 — High-confidence false positive

The model assigned this page a decline probability of approximately 0.887, but the observed label was 0.

The page had 1,266 impressions over the 90-day window, zero recorded clicks, search volume of 0, and 106 days since its last update. The row is also marked as `page_1` with an observed `trend_direction` of `up` and `trend_pct` of +84.8%.

This is a difficult case because the model assigned a very high decline probability despite the observed outcome showing an upward trend. It demonstrates that the learned combination of snapshot features can sometimes strongly disagree with the observed outcome, so the score should be treated as a prioritization signal rather than a guaranteed classification.

In [45]:
false_positives.head(5)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining,decline_probability,predicted_decline,error_type
10175,content_374e795aab68,client_f369cb89fc,880.0,1.0,HIGH,1.67,keyword article,commercial,2675.0,15908.0,...,300.00,0.0,low,page_3_5,stable,0.0,0,0.895327,1,false_positive
27993,content_26d48a980581,client_f369cb89fc,0.0,0.0,LOW,0.00,keyword article,informational,2849.0,20187.0,...,75.00,0.0,moderate,page_1,up,84.8,0,0.886878,1,false_positive
8016,content_c94a53e3bfb8,client_f369cb89fc,0.0,0.0,LOW,0.00,keyword article,informational,2737.0,18468.0,...,133.33,0.0,moderate,page_1,up,71.0,0,0.881281,1,false_positive
22919,content_bf69fff510ad,client_f369cb89fc,10.0,0.0,LOW,0.00,keyword article,transactional,2529.0,17799.0,...,100.00,0.0,moderate,page_1,up,226.2,0,0.861434,1,false_positive
11202,content_ea1fdec27b19,client_f369cb89fc,0.0,0.0,LOW,0.00,keyword article,informational,2675.0,15938.0,...,100.00,0.0,moderate,page_1,up,27.7,0,0.856782,1,false_positive


### 4.5 Error Summary

In [41]:
error_summary = pd.DataFrame({
    "error_type": [
        "False positives",
        "False negatives"
    ],
    "count": [
        (test_results["error_type"] == "false_positive").sum(),
        (test_results["error_type"] == "false_negative").sum()
    ]
})

error_summary

,error_type,count
0,False positives,1798
1,False negatives,938


In [42]:
print("Top permutation-importance features:")
print(
    importance_df.head(5)[
        ["feature", "importance_mean", "importance_std"]
    ]
)

Top permutation-importance features:
                 feature  importance_mean  importance_std
0  days_with_impressions            0.302        0.062897
1            scroll_rate            0.216        0.085697
2           avg_position            0.184        0.026533
3       content_age_days            0.182        0.039446
4           sessions_90d            0.160        0.053666


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.